# Spark Exercises 03 — Spark SQL

Here is something pandas and Polars simply do not have: you can write **plain SQL** against your DataFrames and get a DataFrame back. Register a DataFrame as a *temp view*, then `spark.sql("SELECT ...")`. The SQL engine and the DataFrame API are the *same engine* — you can switch between them freely, mid-pipeline. In Foundry many transforms are written exactly this way.

**Data files:** `data/orders.csv`, `data/customers.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/orders.csv` and `data/customers.csv` (both `header=True`, `inferSchema=True`).

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)

### 3. Register both as **temp views** named `orders` and `customers` using `createOrReplaceTempView`.

In [3]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

### 4. Run your first SQL query: `SELECT * FROM orders LIMIT 5`. Note that `spark.sql(...)` returns a **DataFrame**, so you still call `show()`.

In [4]:
spark.sql("SELECT * FROM orders LIMIT 5").show()

+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|  order_id|customer_id|           order_ts|product_sku|quantity|unit_price|channel|payment_method|   status|discount_code|
+----------+-----------+-------------------+-----------+--------+----------+-------+--------------+---------+-------------+
|ORD-000001|  CUST-0092|2024-07-24 11:01:00|      BK302|      10|     32.58|    web|          card|completed|          VIP|
|ORD-000002|  CUST-0016|2024-05-09 14:53:00|      ST500|      22|      1.43|    web|      applepay|completed|         NULL|
|ORD-000003|  CUST-0289|2024-05-12 20:54:00|      PB024|       9|      1.14|    web|        paypal|completed|         NULL|
|ORD-000004|  CUST-0315|2024-07-07 10:26:00|      MG010|      21|      7.12|    app|          card|completed|         NULL|
|ORD-000005|  CUST-0356|2024-10-15 08:16:00|      PT044|      23|      8.17|  store|        paypal|completed|         NULL|
+-------

### 5. Using SQL, count how many orders have `status = 'completed'`.

In [5]:
spark.sql("SELECT count(*) AS n FROM orders WHERE status = 'completed'").show()

+----+
|   n|
+----+
|6533|
+----+



### 6. Using SQL, compute **revenue per channel** (`sum(quantity*unit_price)`), only for positive quantities, rounded to 2 decimals, highest first.

In [6]:
spark.sql("""
    SELECT channel,
           round(sum(quantity * unit_price), 2) AS revenue
    FROM orders
    WHERE quantity > 0
    GROUP BY channel
    ORDER BY revenue DESC
""").show()

+-------+---------+
|channel|  revenue|
+-------+---------+
|    web|792098.79|
|    app|568985.11|
|  store|349844.45|
|partner|131924.95|
+-------+---------+



### 7. **Mix the two worlds.** Run a SQL query that returns completed orders, keep the result in a Python variable `completed`, then continue with the **DataFrame API** — `completed.select(...).show()`.

In [7]:
completed = spark.sql("SELECT * FROM orders WHERE status = 'completed'")
completed.select("order_id", "channel", "unit_price").show(5)

+----------+-------+----------+
|  order_id|channel|unit_price|
+----------+-------+----------+
|ORD-000001|    web|     32.58|
|ORD-000002|    web|      1.43|
|ORD-000003|    web|      1.14|
|ORD-000004|    app|      7.12|
|ORD-000005|  store|      8.17|
+----------+-------+----------+
only showing top 5 rows



### 8. **JOIN in SQL.** Join `orders` to `customers` on `customer_id` and compute revenue **per customer `segment`** (positive quantities only), highest first.

In [8]:
spark.sql("""
    SELECT c.segment,
           round(sum(o.quantity * o.unit_price), 2) AS revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.quantity > 0
    GROUP BY c.segment
    ORDER BY revenue DESC
""").show()

+----------+----------+
|   segment|   revenue|
+----------+----------+
|  Consumer|1043946.29|
|       SMB| 516710.58|
|Enterprise| 282196.43|
+----------+----------+



### 9. Use a **CTE** (`WITH`) to first compute revenue per customer, then return the **top 5 customers** by revenue.

In [9]:
spark.sql("""
    WITH per_customer AS (
        SELECT customer_id,
               sum(quantity * unit_price) AS revenue
        FROM orders
        WHERE quantity > 0
        GROUP BY customer_id
    )
    SELECT customer_id, round(revenue, 2) AS revenue
    FROM per_customer
    ORDER BY revenue DESC
    LIMIT 5
""").show()

+-----------+--------+
|customer_id| revenue|
+-----------+--------+
|  CUST-0127| 12466.1|
|  CUST-0115|10029.26|
|  CUST-0226| 9729.88|
|  CUST-0287| 9055.15|
|  CUST-0393| 9025.95|
+-----------+--------+



### 10. **SQL inside the DataFrame API.** Use `F.expr(...)` to add a column `revenue = quantity * unit_price` written as a SQL string, and a `CASE WHEN` column `size` ('big' if revenue ≥ 100 else 'small'). Show 5 rows.

In [10]:
orders.withColumn("revenue", F.expr("quantity * unit_price")) \
      .withColumn("size", F.expr("CASE WHEN quantity * unit_price >= 100 THEN 'big' ELSE 'small' END")) \
      .select("order_id", "revenue", "size").show(5)

+----------+------------------+-----+
|  order_id|           revenue| size|
+----------+------------------+-----+
|ORD-000001|325.79999999999995|  big|
|ORD-000002|31.459999999999997|small|
|ORD-000003|             10.26|small|
|ORD-000004|            149.52|  big|
|ORD-000005|            187.91|  big|
+----------+------------------+-----+
only showing top 5 rows



### 11. Register the `completed` DataFrame from earlier as a temp view `completed_orders`, then query it with SQL to count rows per `payment_method`.

In [11]:
completed.createOrReplaceTempView("completed_orders")
spark.sql("""
    SELECT payment_method, count(*) AS n
    FROM completed_orders
    GROUP BY payment_method
    ORDER BY n DESC
""").show()

+--------------+----+
|payment_method|   n|
+--------------+----+
|          card|3585|
|        paypal|1401|
|      applepay|1019|
|      giftcard| 528|
+--------------+----+



### 12. List all the temp views you have registered with `spark.catalog.listTables()`.

In [12]:
spark.catalog.listTables()

[Table(name='completed_orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='customers', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]